# Tests Machine Learning Model

## 📚 Import Libraries

In [1]:
import pandas as pd
import numpy as np
import ast
import re
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

## 💾 Import Data

In [2]:
data = pd.read_csv("../raw_data/recipes_data.csv")

## 🐍 Define Functions

In [3]:
def parse_ner(row):
    try:
        return ast.literal_eval(row)
    except (ValueError, SyntaxError):
        return []

def cosine_similarity(distances):
    return 1 - distances[:, 0].mean()

def coverage(test_set, result_set):
    if not test_set:
        return 0.0
    return len(test_set & result_set) / len(test_set)

def format_results(query_set, distances_row, indices_row, train_df, train_sets, col_str):
    rows = []
    for rank, (j, dist) in enumerate(zip(indices_row, distances_row), 1):
        rec = train_df.iloc[j]
        rows.append({
            "rank": rank,
            "title": rec["title"],
            "ingredients": rec[col_str],
            "cosine_sim": round(1 - dist, 3),
            "coverage": f"{coverage(query_set, train_sets[j]):.1%}",
        })
    return pd.DataFrame(rows).set_index("rank")

def evaluate(distances, indices, train_ingredient_sets, test_ingredient_sets):
    mean_cos = cosine_similarity(distances)
    max_coverages = [
        max(coverage(test_ingredient_sets[i], train_ingredient_sets[j]) for j in neighbors)
        for i, neighbors in enumerate(indices)
    ]
    mean_cov = np.mean(max_coverages)
    personnal_metric = mean_cos + mean_cov
    return mean_cos, mean_cov, personnal_metric


## 1️⃣ First Model — NER 

### ✂ Split Dataset

In [4]:
min_data = data.iloc[:20000].copy()

train_df1, test_df1 = train_test_split(min_data, test_size=0.3, random_state=42)

X1_train = train_df1["NER"]
X1_test  = test_df1["NER"]

print(f"Train : {len(X1_train)} recettes | Test : {len(X1_test)} recettes")

Train : 14000 recettes | Test : 6000 recettes


### ↗ Vectorisation

In [5]:
vectorizer1 = TfidfVectorizer()
X1_train_vec = vectorizer1.fit_transform(X1_train)
X1_test_vec  = vectorizer1.transform(X1_test)

### 🤖 Model

In [6]:
model1 = NearestNeighbors(n_neighbors=5, metric="cosine")
model1.fit(X1_train_vec)

,n_neighbors,5
,radius,1.0
,algorithm,'auto'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,None


### 🧪 Test

In [7]:
distances1, indices1 = model1.kneighbors(X1_test_vec)

### 💯 Score

In [8]:
train1_sets = [set(parse_ner(r)) for r in X1_train.values]
test1_sets  = [set(parse_ner(r)) for r in X1_test.values]

sim1, cov1, metric1 = evaluate(distances1, indices1, train1_sets, test1_sets)
print(f"Model 1 — cosine sim: {sim1:.3f} | coverage top-5: {cov1:.1%} | personnal metric: {metric1:.3f}")


Model 1 — cosine sim: 0.730 | coverage top-5: 66.9% | personnal metric: 1.399


## 2️⃣ Model 2 — Clean Data

### 🧽 Clean

#### 📋 Global Variables

In [9]:
KNOWN_BRANDS = {
    "cool whip", "velveeta", "crisco", "jello", "jell-o", "kraft",
    "bisquick", "campbells", "campbell", "heinz", "hellmanns", "hellmann",
    "karo", "lipton", "miracle whip", "pillsbury", "realemon", "tabasco",
    "worcestershire", "spam", "oreo", "ritz", "swanson", "rotel", "ro-tel",
    "duncan hines", "betty crocker", "knorr", "maggi", "frenchs",
    "hidden valley", "pace", "bushs", "chef boyardee", "progresso",
    "del monte", "green giant", "quaker", "philadelphia",
    "dorito", "doritos", "vegenaise", "squirt", "campbells golden",
}

NON_FOOD = {
    "grill pan", "cookie cutters", "cookie cutter", "cooking spray",
    "wet ingredients", "dry ingredients", "kind deeds", "medley",
    "pan spray", "nonstick spray",
}

MIN_FREQUENCY = 200

#### 🐍 Define Functions

In [10]:
def clean_ingredient(ingredient):
    if not ingredient or pd.isna(ingredient):
        return None
    cleaned = ingredient.strip().lower()
    cleaned = re.sub(r"[^a-z\s\-]", "", cleaned)
    cleaned = re.sub(r"^(a|an|the|some)\s+", "", cleaned).strip()
    if len(cleaned) <= 2:
        return None
    return cleaned

def is_valid(ing):
    if ing in KNOWN_BRANDS or ing in NON_FOOD:
        return False
    if re.search(r'\b(de|of|with|and|or|a|the|in|for)$', ing):
        return False
    if ing.startswith("-") or ing.endswith("-"):
        return False
    return True

In [11]:
ingredient_counter = Counter()
for ner_row in min_data["NER"]:
    for ing in parse_ner(ner_row):
        cleaned = clean_ingredient(ing)
        if cleaned:
            ingredient_counter[cleaned] += 1

valid_ingredients = {
    ing for ing, n in ingredient_counter.items()
    if n >= MIN_FREQUENCY and is_valid(ing)
}

In [12]:
def get_clean_ingredients(ner_row):
    return [
        ing for raw in parse_ner(ner_row)
        if (ing := clean_ingredient(raw)) and ing in valid_ingredients
    ]

min_data = min_data.copy()
min_data["clean_ingredients"] = min_data["NER"].apply(get_clean_ingredients)
min_data["clean_ingredients_str"] = min_data["clean_ingredients"].apply(lambda x: " ".join(x))

print(min_data[["NER", "clean_ingredients"]].head(3).to_string())

                                                                                        NER                                      clean_ingredients
0  ["bite size shredded rice biscuits", "vanilla", "brown sugar", "nuts", "milk", "butter"]             [vanilla, brown sugar, nuts, milk, butter]
1                       ["cream of mushroom soup", "beef", "sour cream", "chicken breasts"]  [cream of mushroom soup, sour cream, chicken breasts]
2              ["frozen corn", "pepper", "cream cheese", "garlic powder", "butter", "salt"]    [pepper, cream cheese, garlic powder, butter, salt]


### ✂ Split Dataset

In [13]:
train_df2, test_df2 = train_test_split(min_data, test_size=0.3, random_state=42)

X2_train = train_df2["clean_ingredients_str"]
X2_test  = test_df2["clean_ingredients_str"]

print(f"Train : {len(X2_train)} recipes | Test : {len(X2_test)} recipes")

Train : 14000 recipes | Test : 6000 recipes


### ↗ Vectorisation

In [14]:
vectorizer2 = TfidfVectorizer()
X2_train_vec = vectorizer2.fit_transform(X2_train)
X2_test_vec  = vectorizer2.transform(X2_test)

### 🤖 Model

In [15]:
model2 = NearestNeighbors(n_neighbors=5, metric="cosine")
model2.fit(X2_train_vec)

,n_neighbors,5
,radius,1.0
,algorithm,'auto'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,None


### 🧪 Test

In [16]:
distances2, indices2 = model2.kneighbors(X2_test_vec)

### 💯 Score

In [17]:
train2_sets = [set(x) for x in train_df2["clean_ingredients"].values]
test2_sets  = [set(x) for x in test_df2["clean_ingredients"].values]

sim2, cov2, metric2 = evaluate(distances2, indices2, train2_sets, test2_sets)
print(f"Model 2 — cosine sim: {sim2:.3f} | coverage top-5: {cov2:.1%} | personnal metric: {metric2:.3f}")


Model 2 — cosine sim: 0.877 | coverage top-5: 84.8% | personnal metric: 1.725


In [21]:
def top5_recipes(sample_idx, distances, indices, train_df, test_sets, train_sets, col_str):
    return format_results(test_sets[sample_idx], distances[sample_idx], indices[sample_idx], train_df, train_sets, col_str)

top5_recipes(42, distances2, indices2, train_df2, test2_sets, train2_sets, "clean_ingredients_str")


,title,ingredients,cosine_sim,coverage
rank,,,,
1,Layered Oatmeal-Chip Cookie Mix,sugar brown sugar baking soda flour chocolate ...,0.871,71.4%
2,Hillary Clinton'S Chocolate Chip Oatmeal Cookies,sugar vanilla baking soda brown sugar eggs flo...,0.867,85.7%
3,Marshmallow Cloud Cookies,sugar vanilla baking soda brown sugar eggs flo...,0.858,85.7%
4,Chop Suey Cookies,oleo sugar vanilla baking powder brown sugar e...,0.858,100.0%
5,Blonde Brownies(Shirley Mckenzie),oleo baking powder vanilla brown sugar eggs ba...,0.857,85.7%


In [19]:
def predict(ingredients, model, vectorizer, train_df, train_sets, valid_ingredients, col_str, n=5):
    cleaned = [c for raw in ingredients if (c := clean_ingredient(raw)) and c in valid_ingredients]
    if not cleaned:
        print("No valid ingredients found after cleaning.")
        return None

    query_str = " ".join(cleaned)
    query_vec = vectorizer.transform([query_str])
    distances, indices = model.kneighbors(query_vec, n_neighbors=n)

    return format_results(set(cleaned), distances[0], indices[0], train_df, train_sets, col_str)


my_ingredients = ["chicken", "garlic", "lemon", "olive oil"]

predict(my_ingredients, model2, vectorizer2, train_df2, train2_sets, valid_ingredients, "clean_ingredients_str")


,title,ingredients,cosine_sim,coverage
rank,,,,
1,Italian Sauteed Chicken With Italian Mushroom ...,flour chicken garlic butter olive oil,0.913,100.0%
2,Roasted Red Peppers,garlic olive oil,0.894,66.7%
3,Garlic Olives,garlic olive oil,0.894,66.7%
4,Garlic Green Beans,garlic olive oil,0.894,66.7%
5,Cicoria Fina Agliata(Dandelion Sauteed With Ga...,garlic olive oil,0.894,66.7%


## 📊 Compare

In [22]:
comparison = pd.DataFrame(
    {
        "Model 1 — NER": [round(sim1, 3), f"{cov1:.1%}", f"{metric1:.3f}"],
        "Model 2 — Clean Ingredients": [round(sim2, 3), f"{cov2:.1%}", f"{metric2:.3f}"],
    },
    index=["Cosine similarity top-1", "Coverage Score top-5", "Cosine + Coverage"],
)
comparison


,Model 1 — NER,Model 2 — Clean Ingredients
Cosine similarity top-1,0.73,0.877
Coverage Score top-5,66.9%,84.8%
Cosine + Coverage,1.399,1.725
